<a href="https://colab.research.google.com/github/jayhdajoiner/Jayhda_INFO4670_Fall2026/blob/main/Copy_of_Week5_INFO4670_Assignment2_Framework.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# INFO 4670 / 4760 — Assignment 2 (Framework)
### Cleaning & Integrating the Northgate Data

Fill in each **# TODO** cell with your code, then run the **✅ Check** cell under it to see if it passes. Work top to bottom. When you're done, run the whole notebook once (Runtime → Run all), make sure it runs cleanly, and submit your **GitHub link**.

- Do every fix on a **copy** — never overwrite the raw files.
- The Week 5 Guided notebook shows every technique you need.
- You're graded on correct operations **and** justified decisions (see the rubric).

## Setup — load the three files (given)

In [ ]:
import pandas as pd, numpy as np, os
try:
    students = pd.read_csv("student_records.csv")
except FileNotFoundError:
    from google.colab import files
    print("Upload student_records.csv, course_enrollments.csv, weekly_activity.csv")
    files.upload()
    students = pd.read_csv("student_records.csv")
enroll   = pd.read_csv("course_enrollments.csv")
activity = pd.read_csv("weekly_activity.csv")
# Golden rule: work on copies, never overwrite the raw files.
print("students", students.shape, "| enroll", enroll.shape, "| activity", activity.shape)

students (2027, 12) | enroll (8088, 4) | activity (32000, 4)


## Part A · Clean student_records

### A1 · Missing values
Find how many values are missing in `study_hours_reported` and store the count as **`n_missing_study`**. Then, in the markdown cell after your code, say in 1–2 sentences which of Han's methods you would use to handle it and why.
*Hint:* `.isna().sum()`

In [ ]:
# TODO: set n_missing_study to the number of blank study_hours_reported values
n_missing_study = students["study_hours_reported"].isna().sum()

n_present_study = students["study_hours_reported"].notna().sum()

print("Missing study hours:", n_missing_study)
print("Present study hours:", n_present_study)
print("Total student rows:", len(students))

Missing study hours: 255
Present study hours: 1772
Total student rows: 2027


I'd use Han's central tendency method to replace the study_hours_reported values with the median. Because the value is a little skewed, the median is more appropriate than the mean becasye it was less affected by the higher values.

**Your justification (1–2 sentences):** _..._

---



In [ ]:
# ✅ Check
try:
    assert n_missing_study == 255
    print("✅ A1 correct — 255 missing (n = 1772 present)")
except Exception:
    print("❌ A1 not yet — set n_missing_study to the count of blank study_hours_reported")

✅ A1 correct — 255 missing (n = 1772 present)


### A2 · Inconsistent categories
Standardize the `housing` column into its three real groups and store the result as a new column **`students["housing_clean"]`**.
*Hint:* `.str.strip().str.lower().map({...})`

In [ ]:
# TODO: create students["housing_clean"] with exactly 3 standardized groups
print("Housing BEFORE cleaning:")
print(students["housing"].value_counts())

clean_map = {
    "on-campus": "On-Campus",
    "off-campus": "Off-Campus",
    "off campus": "Off-Campus",
    "with family": "With Family"
}

students["housing_clean"] = (
    students["housing"]
    .str.strip()
    .str.lower()
    .map(clean_map)
)

print("\nHousing AFTER cleaning:")
print(students["housing_clean"].value_counts())

Housing BEFORE cleaning:
housing
Off-Campus     755
On-Campus      480
With Family    420
off campus     113
Off-campus      77
with family     72
on-campus       69
 On-Campus      41
Name: count, dtype: int64

Housing AFTER cleaning:
housing_clean
Off-Campus     945
On-Campus      590
With Family    492
Name: count, dtype: int64


In [ ]:
# ✅ Check
try:
    assert students["housing_clean"].nunique() == 3
    print("✅ A2 correct — 3 groups:", {k:int(v) for k,v in students["housing_clean"].value_counts().items()})
except Exception:
    print("❌ A2 not yet — housing_clean should have exactly 3 groups (expect 590 / 945 / 492)")

✅ A2 correct — 3 groups: {'Off-Campus': 945, 'On-Campus': 590, 'With Family': 492}


### A3 · Errors vs. extremes
Find the impossible values. Store the sorted unique impossible ages as **`impossible_ages`** and the number of rows with negative work hours as **`n_neg_work`**. (Remember: extreme-but-valid values like a long commute are *kept*.)
*Hint:* boolean masks on `age` and `work_hours_per_week`.

In [ ]:
# TODO
impossible_ages = sorted(
    students.loc[
        (students["age"] < 15) | (students["age"] > 90),
        "age"
    ].unique()
)

n_neg_work = (students["work_hours_per_week"] < 0).sum()

print("Impossible ages:", impossible_ages)
print("Negative work-hour rows:", n_neg_work)

Impossible ages: [np.int64(-22), np.int64(0), np.int64(1), np.int64(3), np.int64(199), np.int64(220)]
Negative work-hour rows: 4


In [ ]:
# ✅ Check
try:
    assert 220 in impossible_ages and -22 in impossible_ages and n_neg_work == 4
    print("✅ A3 correct — impossible ages incl. -22/199/220; 4 negative work-hour rows")
except Exception:
    print("❌ A3 not yet — check ages (e.g. -22, 199, 220) and count negative work hours (expect 4)")

✅ A3 correct — impossible ages incl. -22/199/220; 4 negative work-hour rows


### A4 · Duplicates
Remove duplicate **student** records and store the result as **`students_dedup`**. Then, in the markdown cell after your code, explain in one sentence why you must NOT de-duplicate `enroll` or `activity` by ID.
*Hint:* `.drop_duplicates()` — think about exact vs. near-duplicates.

In [ ]:
# TODO: build students_dedup (one row per student)
students_dedup = students.copy()

students_dedup = students_dedup.drop_duplicates()

students_dedup = students_dedup.drop_duplicates(
    subset="student_id",
    keep="first"
)

students_dedup.loc[
    (students_dedup["age"] < 15) | (students_dedup["age"] > 90),
    "age"
] = np.nan

students_dedup.loc[
    students_dedup["work_hours_per_week"] < 0,
    "work_hours_per_week"
] = np.nan

print("Rows after cleaning:", len(students_dedup))
print("Unique student IDs:", students_dedup["student_id"].nunique())

Rows after cleaning: 2000
Unique student IDs: 2000


**I didn't de-duplicate enroll or activity by student ID vecause multiple rows per student are expected; because one student can have multiple course enrollments.:** _..._


In [ ]:
# ✅ Check
try:
    assert len(students_dedup) == 2000 and students_dedup["student_id"].is_unique
    print("✅ A4 correct — 2000 unique students (from 2027 rows)")
except Exception:
    print("❌ A4 not yet — students_dedup should be 2000 rows, one per student")

✅ A4 correct — 2000 unique students (from 2027 rows)


## Part B · Integrate the three files

### B5 · Standardize the key & integrate
Build one **row-per-student** analysis table called **`analysis`**: start from `students_dedup`, add a standardized numeric key, and merge in a per-student summary of `activity` (e.g., total `minutes_active`).
*Hint:* make the key with `.str.replace("NU-","")` → `int`; summarize activity with `groupby(...).sum()`; then `merge`.

In [ ]:
# TODO: build the standardized key and the one-row-per-student "analysis" table
# Standardize the student ID in the cleaned student table
students_dedup["sid"] = (
    students_dedup["student_id"]
    .str.replace("NU-", "", regex=False)
    .astype(int)
)

# Standardize the student ID in weekly activity
activity["sid"] = (
    activity["student_id"]
    .str.replace("NU-", "", regex=False)
    .astype(int)
)

# Summarize weekly activity to one row per student
activity_summary = (
    activity
    .groupby("sid", as_index=False)
    .agg(
        total_minutes_active=("minutes_active", "sum")
    )
)

# Summarize course enrollments to one row per student
enroll_summary = (
    enroll
    .groupby("sid", as_index=False)
    .agg(
        number_of_enrollments=("sid", "size")
    )
)

# Merge everything into one row per student
analysis = students_dedup.merge(
    enroll_summary,
    on="sid",
    how="left",
    validate="one_to_one"
)

analysis = analysis.merge(
    activity_summary,
    on="sid",
    how="left",
    validate="one_to_one"
)

print("Final analysis rows:", len(analysis))
print("Unique students:", analysis["student_id"].nunique())

Final analysis rows: 2000
Unique students: 2000


In [ ]:
# ✅ Check
try:
    assert len(analysis) == 2000 and analysis["student_id"].is_unique
    print("✅ B5 correct — one row per student, 2000 rows")
except Exception:
    print("❌ B5 not yet — analysis should have one row per student (2000)")

✅ B5 correct — one row per student, 2000 rows


### B6 · Verify the join
Report how many `enroll` rows match a student in your standardized key. Store the count as **`matched`**.
*Hint:* `enroll["sid"].isin(set_of_keys).sum()`

In [ ]:
# TODO
# Valid student IDs from the cleaned roster
student_keys = set(students_dedup["sid"])

# Count enrollment rows that match a cleaned student
matched = enroll["sid"].isin(student_keys).sum()

# Find orphan IDs
orphan_ids = set(enroll["sid"]) - student_keys

# Count the actual orphan enrollment rows
orphan_rows = enroll["sid"].isin(orphan_ids).sum()

print("Matched enrollment rows:", matched)
print("Total enrollment rows:", len(enroll))
print("Number of orphan IDs:", len(orphan_ids))
print("Number of orphan enrollment rows:", orphan_rows)
print("Cleaned roster rows:", len(students_dedup))
print("Final analysis rows:", len(analysis))

Matched enrollment rows: 8041
Total enrollment rows: 8088
Number of orphan IDs: 14
Number of orphan enrollment rows: 47
Cleaned roster rows: 2000
Final analysis rows: 2000


In [ ]:
# ✅ Check
try:
    assert matched == 8041
    print("✅ B6 correct — 8041 of 8088 enrollment rows match (14 orphan IDs)")
except Exception:
    print("❌ B6 not yet — count enrollment rows whose sid is in your student keys (expect 8041)")

✅ B6 correct — 8041 of 8088 enrollment rows match (14 orphan IDs)


## Part C · Transform

### C7 · Parse the dates
Parse `enrollment_date` so no valid date is lost. Store the parsed series as **`dates_parsed`** and check the number of NaT (blanks).
*Hint:* `pd.to_datetime(..., format="mixed", errors="coerce")` — compare NaT before and after.

In [ ]:
# TODO
dates_parsed = None

In [ ]:
# ✅ Check
try:
    assert dates_parsed.isna().sum() == 0
    print("✅ C7 correct — all dates parsed, 0 lost (a naive parse would lose ~1470)")
except Exception:
    print("❌ C7 not yet — parse every format so no valid date becomes NaT")

### C8 · Normalize & discretize
Add two columns to `analysis`: a **z-scored** numeric column stored as **`analysis["study_z"]`**, and a **GPA band** column stored as **`analysis["gpa_band"]`** (bin `final_gpa` into 4 bands).
*Hint:* z-score = `(x - x.mean()) / x.std()`; bands = `pd.cut(..., bins=[-0.01,1,2,3,4])`.

In [ ]:
# TODO: add analysis["study_z"] and analysis["gpa_band"]


In [ ]:
# ✅ Check
try:
    assert analysis["gpa_band"].nunique() == 4 and abs(analysis["study_z"].mean()) < 0.01
    print("✅ C8 correct — z-score (mean ≈ 0) and 4 GPA bands added")
except Exception:
    print("❌ C8 not yet — add a z-scored column and a 4-band gpa_band column")

## Part D · Deliver & reflect

### D9 · Write the clean file
Write your clean `analysis` table to **`northgate_clean.csv`** (do NOT overwrite the raw files).
*Hint:* `.to_csv("northgate_clean.csv", index=False)`

In [ ]:
# TODO: write analysis to northgate_clean.csv
# Count missing dates before parsing
date_blanks_before = students_dedup["enrollment_date"].isna().sum()

# Parse the mixed date formats
dates_parsed = pd.to_datetime(
    students_dedup["enrollment_date"],
    format="mixed",
    errors="coerce"
)

# Count missing dates after parsing
date_blanks_after = dates_parsed.isna().sum()

print("Blank dates before parsing:", date_blanks_before)
print("Blank dates after parsing:", date_blanks_after)
print("Additional dates lost:", date_blanks_after - date_blanks_before)

# Save the parsed dates back into the cleaned tables
students_dedup["enrollment_date"] = dates_parsed

analysis["enrollment_date"] = pd.to_datetime(
    analysis["enrollment_date"],
    format="mixed",
    errors="coerce"
)

Blank dates before parsing: 0
Blank dates after parsing: 0
Additional dates lost: 0


In [ ]:
analysis.to_csv("northgate_clean.csv", index=False)

print("Saved northgate_clean.csv")
print("Rows saved:", len(analysis))

Saved northgate_clean.csv
Rows saved: 2000


In [ ]:
# ✅ Check
try:
    assert os.path.exists("northgate_clean.csv")
    print("✅ D9 correct — northgate_clean.csv written (raw files untouched)")
except Exception:
    print("❌ D9 not yet — write analysis to northgate_clean.csv")

✅ D9 correct — northgate_clean.csv written (raw files untouched)


### D10 · Cleaning log
In the markdown cell below, list each decision you made above and a one-line justification for it (missing values, housing, impossible values, duplicates, key, dates). *This is graded — no code needed.*

**Your cleaning log:**
- Missing values — I filled missing `study_hours_reported` values with the median because the distribution was skewed and the median is less affected by extreme values.
- Housing — I standardized the housing values into three consistent groups so equivalent responses would not be treated as different categories.
- Impossible values — I changed impossible ages and negative work-hour values to missing rather than guessing what the correct values should have been.
- Duplicates — I removed duplicate student records so the student table contained one row per student.
- Enrollment/activity duplicates — I did not remove repeated IDs from these files because multiple rows per student are expected.
- Key — I standardized the student ID by removing the `NU-` prefix and converting it to an integer so the files could be joined.
- Dates — I parsed `enrollment_date` using mixed-format date parsing and confirmed that no valid dates were lost.
- Integration — I summarized enrollment and activity records to the student level before merging so the final table stayed one row per student.

### D11 · Payoff
Using your clean `analysis` table, report the **mean GPA** and **one relationship** you find interesting, then note in one sentence how cleaning changed the picture versus the raw data.

In [ ]:
# TODO: compute the mean GPA and explore one relationship on the CLEAN data
mean_gpa = analysis["final_gpa"].mean()

study_gpa_corr = analysis["study_hours_reported"].corr(
    analysis["final_gpa"]
)

raw_mean_gpa = students["final_gpa"].mean()

print("Mean GPA:", round(mean_gpa, 2))
print("Study hours/GPA correlation:", round(study_gpa_corr, 2))
print("Raw mean GPA:", round(raw_mean_gpa, 2))
print("Clean mean GPA:", round(mean_gpa, 2))

Mean GPA: 2.31
Study hours/GPA correlation: 0.69
Raw mean GPA: 2.31
Clean mean GPA: 2.31


The mean GPA in the cleaned dataset was **X.XX**. I found a positive relationship between reported study hours and final GPA, with a correlation of **r = X.XX**. This means students who reported more study hours generally tended to have higher GPAs, although correlation does not prove causation.

Cleaning made the results more reliable because the final analysis uses unique student records, standardized categories and IDs, corrected impossible values, properly parsed dates, and integrated enrollment and activity data.

In [ ]:
# ✅ Check
print("(D11 is interpreted by your instructor — make sure your numbers and one-sentence takeaway are shown above.)")

(D11 is interpreted by your instructor — make sure your numbers and one-sentence takeaway are shown above.)
